<a href="https://colab.research.google.com/github/docling-project/docling/blob/main/docs/examples/rag_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG with LangChain

| Step | Tech | Execution | 
| --- | --- | --- |
| Embedding | Hugging Face / Sentence Transformers | 💻 Local |
| Vector store | Milvus | 💻 Local |
| Gen AI | Groq API | 🌐 Remote | 

This example leverages the
[LangChain Docling integration](../../integrations/langchain/), along with a Milvus
vector store, as well as sentence-transformers embeddings.

The presented `DoclingLoader` component enables you to:
- use various document types in your LLM applications with ease and speed, and
- leverage Docling's rich format for advanced, document-native grounding.

`DoclingLoader` supports two different export modes:
- `ExportType.MARKDOWN`: if you want to capture each input document as a separate
  LangChain document, or
- `ExportType.DOC_CHUNKS` (default): if you want to have each input document chunked and
  to then capture each individual chunk as a separate LangChain document downstream.

The example allows exploring both modes via parameter `EXPORT_TYPE`; depending on the
value set, the example pipeline is then set up accordingly.

## Setup

- 👉 For best conversion speed, use GPU acceleration whenever available; e.g. if running on Colab, use GPU-enabled runtime.
- Notebook uses Groq's OpenAI-compatible API.
- Requirements can be installed as shown below (`--no-warn-conflicts` meant for Colab's pre-populated Python env; feel free to remove for stricter usage):

In [1]:
%pip install -q --progress-bar off --no-warn-conflicts langchain-docling langchain-core langchain-huggingface langchain_milvus langchain python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
from pathlib import Path
from tempfile import mkdtemp

from langchain_core.prompts import PromptTemplate
from langchain_docling.loader import ExportType
from dotenv import load_dotenv

load_dotenv()

# https://github.com/huggingface/transformers/issues/5486:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

FILE_PATH = ["https://arxiv.org/pdf/2408.09869"]  # Docling Technical Report
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
EXPORT_TYPE = ExportType.DOC_CHUNKS
QUESTION = "Which are the main AI models in Docling?"
PROMPT = PromptTemplate.from_template(
    "Context information is below.\n---------------------\n{context}\n---------------------\nGiven the context information and not prior knowledge, answer the query.\nQuery: {input}\nAnswer:\n",
)
TOP_K = 3
MILVUS_URI = str(Path(mkdtemp()) / "docling.db")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LLM = "openai/gpt-oss-20b"
BASE_URL = "https://api.groq.com/openai/v1"

## Document loading

Now we can instantiate our loader and load documents.

In [14]:
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from langchain_docling import DoclingLoader
from transformers import AutoTokenizer

from docling.chunking import HybridChunker

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID)
)

loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=EXPORT_TYPE,
    chunker=HybridChunker(tokenizer=tokenizer),
)

docs = loader.load()

[INFO] 2026-10-05 12:16:30,618 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-05 12:16:30,647 [RapidOCR] download_file.py:60: File exists and is valid: /home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-10-05 12:16:30,648 [RapidOCR] main.py:63: Using /home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-10-05 12:16:31,027 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-05 12:16:31,030 [RapidOCR] download_file.py:60: File exists and is valid: /home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-05 12:16:31,030 [RapidOCR] main.py:63: Using /home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-05 12:16:31,093 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

RapidOCR returned empty result!


2026-10-05 12:17:12,770 MatchingPostProcessor WARNING  Orphan pdf_cell 122 recovered to row=3 by nearest-row fallback (col=0, y=490.5, dist=48.6)
2026-10-05 12:17:12,771 MatchingPostProcessor WARNING  Orphan pdf_cell 123 recovered to row=3 by nearest-row fallback (col=0, y=490.5, dist=48.6)
2026-10-05 12:17:57,920 MatchingPostProcessor WARNING  Orphan pdf_cell 200 recovered to col=4 by nearest-column fallback (row=0, x=908.3, dist=26.3)
2026-10-05 12:17:57,921 MatchingPostProcessor WARNING  Orphan pdf_cell 201 recovered to col=4 by nearest-column fallback (row=0, x=910.0, dist=28.0)
2026-10-05 12:17:57,922 MatchingPostProcessor WARNING  Orphan pdf_cell 215 recovered to col=4 by nearest-column fallback (row=2, x=908.0, dist=26.0)
2026-10-05 12:17:57,922 MatchingPostProcessor WARNING  Orphan pdf_cell 216 recovered to col=4 by nearest-column fallback (row=1, x=908.3, dist=26.3)
2026-10-05 12:17:57,922 MatchingPostProcessor WARNING  Orphan pdf_cell 223 recovered to col=4 by nearest-column 

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (708 > 512). Running this sequence through the model will result in indexing errors


> Note: a message saying `"Token indices sequence length is longer than the specified
maximum sequence length..."` can be ignored in this case — details
[here](https://github.com/docling-project/docling-core/issues/119#issuecomment-2577418826).

Determining the splits:

In [15]:
if EXPORT_TYPE == ExportType.DOC_CHUNKS:
    splits = docs
elif EXPORT_TYPE == ExportType.MARKDOWN:
    from langchain_text_splitters import MarkdownHeaderTextSplitter

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header_1"),
            ("##", "Header_2"),
            ("###", "Header_3"),
        ],
    )
    splits = [split for doc in docs for split in splitter.split_text(doc.page_content)]
else:
    raise ValueError(f"Unexpected export type: {EXPORT_TYPE}")

Inspecting some sample splits:

In [16]:
for d in splits[:3]:
    print(f"- {d.page_content=}")
print("...")

- d.page_content='Version 1.0\nChristoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar\nAI4K Group, IBM Research R¨ uschlikon, Switzerland'
- d.page_content='Abstract\nThis technical report introduces Docling , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility and addition of new features and models.'
- d.page_content='1 Introduction\nConverting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge vari

## Ingestion

In [17]:
import json
from pathlib import Path
from tempfile import mkdtemp

from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_milvus import Milvus
from pymilvus import connections

embedding = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID)


milvus_uri = str(Path(mkdtemp()) / "docling.db")  # or set as needed
vectorstore = Milvus(
    embedding_function=embedding,
    collection_name="docling_demo",
    connection_args={"uri": milvus_uri},
    # Milvus Lite (milvus-lite 3.x) requires an explicit metric_type;
    # COSINE suits the normalized sentence-transformers embeddings.
    index_params={"index_type": "FLAT", "metric_type": "COSINE"},
    drop_old=True,
    auto_id=True,
)
# pymilvus 2.6's MilvusClient no longer registers an ORM-style connection, but
# langchain-milvus 0.3.x still builds an ORM Collection under this alias, so we
# register it explicitly to avoid ConnectionNotExistException.
connections.connect(alias=vectorstore.alias, uri=milvus_uri)
vectorstore.add_documents(splits)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_75658/1930299213.py:26: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(alias=vectorstore.alias, uri=milvus_uri)
/home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/langchain_milvus/vectorstores/milvus.py:580: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  self._col_cache = Collection(self.collection_name, using=self.alias)
/home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/langchain_milvus/vectorstores/milvus.py:1041: PyMilvusDeprecationWarning: `Collection.indexes` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  for x in self.col.indexes:
/home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/langchain_milvus/vectorstores/milvus.py:1043: PyMilvusDeprecationWarning: `Index.to_dict` is 

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49]

## RAG

In [18]:
from langchain_openai import ChatOpenAI

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=GROQ_API_KEY,
    model=LLM,
    temperature=0
)


def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

I1005 12:22:23.079194   80106 chttp2_transport.cc:1402] ipv4:127.0.0.1:40545: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1005 12:22:23.082361   80106 chttp2_transport.cc:1434] ipv4:127.0.0.1:40545: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


def _format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


# langchain 1.x removed the legacy create_retrieval_chain /
# create_stuff_documents_chain helpers (now only in the separate langchain-classic
# package). This LCEL chain is the modern equivalent and returns the same
# {"input", "context", "answer"} structure the printing below relies on.
rag_chain = RunnablePassthrough.assign(
    context=lambda x: retriever.invoke(x["input"]),
).assign(
    answer=(
        RunnableLambda(
            lambda x: {"input": x["input"], "context": _format_docs(x["context"])}
        )
        | PROMPT
        | llm
        | StrOutputParser()
    )
)
resp_dict = rag_chain.invoke({"input": QUESTION})

clipped_answer = clip_text(resp_dict["answer"], threshold=200)
print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clipped_answer}")
for i, doc in enumerate(resp_dict["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
    for key in doc.metadata:
        if key != "pk":
            val = doc.metadata.get(key)
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")

Question:
Which are the main AI models in Docling?

Answer:
The core AI models that Docling ships with are:

| Model | Purpose | Key Features |
|-------|---------|--------------|
| **Layout Analysis Model** | Detects and classifies page elements (e.g., heading...

Source 1:
  text: "3.2 AI models\nAs part of Docling, we initially release two highly capable AI models to the open-source community, which have been developed and published recently by our team. The first model is a layout analysis model, an accurate object-detector for page elements [13]. The second model is TableFormer [12, 9], a state-of-the-art table structure re..."
  source: https://arxiv.org/pdf/2408.09869
  dl_meta: {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/50', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 3, 'bbox': {'l': 108.0, 't': 404.8726526000001, 'r': 504.00338739999984, 'b':

I1005 12:27:34.445124   80107 chttp2_transport.cc:1402] ipv4:127.0.0.1:40545: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1005 12:27:34.446104   80107 chttp2_transport.cc:1434] ipv4:127.0.0.1:40545: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 20000ms
